# Lab 2 - Modelos Tradicionais

Classificadores: **KNN**, **Árvore de Decisão** e **SVM**.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from lab2_utils import (
    carregar_dados, avaliar_modelo, logar_mlflow, iniciar_run,
    calibrar_threshold, predizer_com_threshold,
)

X_train, y_train, X_test, y_test = carregar_dados()

# Split de validação dentro do treino — usado para calibrar threshold
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

Iniciando servidor MLflow em background...


Servidor MLflow pronto em http://127.0.0.1:5000


Dados carregados (gerados em 2026-06-25 21:29)
  X_train: (440832, 10)  |  X_test: (64374, 10)
  Features: ['Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Total Spend', 'Last Interaction', 'Subscription Type_Premium', 'Subscription Type_Standard']
  Churn rate treino: 0.567  |  teste: 0.474


## KNN

In [2]:
# ─── KNN ────────────────────────────────────────────────────────
# Treina em X_tr (80% do treino), calibra threshold via predict_proba
# no X_val (20%), prediz no X_test com threshold ótimo.
from sklearn.neighbors import KNeighborsClassifier

params = {
    'modelo': 'KNN',
    'n_neighbors': 11,
    'weights': 'distance',
    'metric': 'minkowski',
    'scaler': 'StandardScaler (centralizado)',
}

with iniciar_run("KNN", notebook="2A", params=params):
    model = KNeighborsClassifier(
        n_neighbors=11,
        weights='distance',
        n_jobs=-1,
    )
    model.fit(X_tr, y_tr)

    threshold = calibrar_threshold(model, X_val, y_val)
    y_pred = predizer_com_threshold(model, X_test, threshold)
    params['threshold'] = round(threshold, 4)
    print(f'Threshold calibrado: {threshold:.4f}')

    metricas = avaliar_modelo('KNN', y_test, y_pred)
    logar_mlflow(metricas, params)

Threshold calibrado: 0.3393

=== KNN ===
              precision    recall  f1-score   support

Não cancelou       0.95      0.14      0.25     33881
    Cancelou       0.51      0.99      0.67     30493

    accuracy                           0.55     64374
   macro avg       0.73      0.57      0.46     64374
weighted avg       0.74      0.55      0.45     64374

Balanced Accuracy: 0.5675  |  Kappa: 0.1290


🏃 View run KNN at: http://127.0.0.1:5000/#/experiments/1/runs/0a97542082e7441d919d7d3f7c7f6215
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## Árvore de Decisão

In [3]:
# ─── Árvore de Decisão ──────────────────────────────────────────
# Threshold calibrado via predict_proba substitui o ajuste por class_weight.
from sklearn.tree import DecisionTreeClassifier

params = {
    'modelo': 'DecisionTree',
    'max_depth': 5,
    'min_samples_split': 50,
    'min_samples_leaf': 20,
    'criterion': 'gini',
    'scaler': 'StandardScaler (centralizado)',
}

with iniciar_run("DecisionTree", notebook="2A", params=params):
    model = DecisionTreeClassifier(
        max_depth=5,
        min_samples_split=50,
        min_samples_leaf=20,
        criterion='gini',
        random_state=42,
    )
    model.fit(X_tr, y_tr)

    threshold = calibrar_threshold(model, X_val, y_val)
    y_pred = predizer_com_threshold(model, X_test, threshold)
    params['threshold'] = round(threshold, 4)
    print(f'Threshold calibrado: {threshold:.4f}')

    metricas = avaliar_modelo('DecisionTree', y_test, y_pred)
    logar_mlflow(metricas, params)

Threshold calibrado: 1.0000

=== DecisionTree ===
              precision    recall  f1-score   support

Não cancelou       0.98      0.16      0.27     33881
    Cancelou       0.52      1.00      0.68     30493

    accuracy                           0.55     64374
   macro avg       0.75      0.58      0.47     64374
weighted avg       0.76      0.55      0.46     64374

Balanced Accuracy: 0.5761  |  Kappa: 0.1454


🏃 View run DecisionTree at: http://127.0.0.1:5000/#/experiments/1/runs/8a1e100110504103a634603ad8ee33b8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## SVM

In [4]:
# ─── SVM ────────────────────────────────────────────────────────
# CalibratedClassifierCV adiciona predict_proba ao LinearSVC,
# permitindo calibração de threshold via curva PR.
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

params = {
    'modelo': 'SVM',
    'tipo': 'CalibratedLinearSVC',
    'C': 0.1,
    'class_weight': 'balanced',
    'max_iter': 2000,
    'scaler': 'StandardScaler (centralizado)',
}

with iniciar_run("SVM", notebook="2A", params=params):
    base_svm = LinearSVC(
        C=0.1,
        class_weight='balanced',
        max_iter=2000,
        random_state=42,
    )
    model = CalibratedClassifierCV(base_svm, cv=3)
    model.fit(X_tr, y_tr)

    threshold = calibrar_threshold(model, X_val, y_val)
    y_pred = predizer_com_threshold(model, X_test, threshold)
    params['threshold'] = round(threshold, 4)
    print(f'Threshold calibrado: {threshold:.4f}')

    metricas = avaliar_modelo('SVM', y_test, y_pred)
    logar_mlflow(metricas, params)

Threshold calibrado: 0.4861

=== SVM ===
              precision    recall  f1-score   support

Não cancelou       0.94      0.22      0.36     33881
    Cancelou       0.53      0.98      0.69     30493

    accuracy                           0.58     64374
   macro avg       0.74      0.60      0.53     64374
weighted avg       0.75      0.58      0.52     64374

Balanced Accuracy: 0.6040  |  Kappa: 0.1995


🏃 View run SVM at: http://127.0.0.1:5000/#/experiments/1/runs/09be631ed4da48d0ae16226a87706733
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
